# A3a -- structured ambient RNA, made visible and then bounded

Answers **Reviewer #2, major point 9** (`plans/Round2_response_analysis_plan.md` section A3).

> "The ambient background is modeled as complete spatial randomness (CSR), yet ambient RNA from
> debris, dying cells, and extracellular vesicles is typically spatially structured rather than
> uniform, and is likely to be denser precisely where cells are denser, or in regions of severe AD
> pathology... A CSR-based threshold could therefore under-correct in such regions and inflate
> granule calls locally... The downstream density regression is reassuring but does not test this,
> since it operates on granules that have already been called. A direct check at the detection
> step, such as the pseudo-granule negative control I suggested, would settle the question."

### The idea

The reviewer names the sources: **debris, dying cells, extracellular vesicles**. All three release
*somatic* RNA into the extracellular space. So we can make that population visible by running the
detector on genes that have no legitimate reason to be outside a soma, and keeping only the
aggregates that are outside one.

The panel's **negative-control genes** are exactly those: an edgeR list of transcripts enriched in
neuronal nuclei relative to cytoplasm (manuscript Supp. Table 8). We run mcDETECT on them, apply
the in-soma filter, and apply **no negative-control filter** -- those genes are now the seeds, so
filtering on them would be meaningless. What survives is **Set 3: extrasomatic aggregates of
soma-restricted transcripts**, i.e. structured ambient RNA itself.

Set 3 is then compared against the granule populations at two stages: **Set 1**, the markers put
through the same pipeline *before* negative-control filtering, and **Set 2**, the published
granules.

### The four sets

| set | seeds on | filters applied | source |
|---|---|---|---|
| **Set 0** | 20 panel genes with no synaptic / neuropil / control annotation, abundance-matched to the markers | size + in-soma | new |
| **Set 1** | the 20 granule markers | size + in-soma, **no NC filter** | new |
| **Set 2** | the 20 granule markers | size + in-soma + NC filter | **published** |
| **Set 3** | the 18 negative-control genes | size + in-soma, **no NC filter** | new |

**Set 3 carries the ambient interpretation. Set 0 carries the abundance interpretation.** Set 3's
genes are ~15x rarer than the markers (median 74,893 transcripts against 1,153,633), and DBSCAN
yield is superlinear in count, so "Set 3 is small" could be explained by rarity alone. Set 0 is the
answer to that: arbitrary genes at matched abundance through the identical pipeline. Both are run
through all four claims, and both are scored against Set 1 *and* Set 2.

Two things about Set 0 that must be stated wherever it is used, not buried:

* the abundance match is exact at the bottom and poor at the top -- the unannotated pool holds no
  gene above ~300 K transcripts, so `Camk2a` (6.24 M) is matched to `Zbtb20` (267 K), a 23x gap;
* "unannotated" is not "non-dendritic". Four of the twenty -- `Grin2b`, `Dner`, `Epha4`, `Ncam1` --
  are documented dendritically-localized transcripts, so a *higher* Set 0 yield is expected and is
  not by itself evidence of ambient contamination.

Every set is reported as a **funnel** (raw -> size -> in-soma) rather than an endpoint, so the
stage at which each population thins is visible rather than asserted.

### The Gria2 policy

`Gria2` is on both the 20-gene marker list and the 19-gene control list. The policy is fixed and
not analysed here: the **marker seed list is never modified** (Sets 1 and 2 seed on all 20);
**Set 3 seeds on 18**, because seeding a control population on a canonical dendritic marker would
manufacture the overlap Set 3 exists to bound; **Set 2 is read exactly as published**, on the
19-gene filter it was actually built with. The residual discrepancy is under half a percent of
Set 1, in the same direction in both samples.

### The four claims this notebook produces

| § | claim | writes |
|---|---|---|
| 1 | structured ambient RNA exists, and it is **sparse** | `set_inventory.csv`, `funnel_by_gene.csv` |
| 2 | control aggregates **rarely coincide** with granules, and the NC filter removes most of those that do | `overlap_ladder.csv`, `overlap_transcript_level.csv`, `set2_reproduction.csv` |
| 3 | control density **cannot reproduce** the WT-vs-AD result | `set_density_per_region.csv`, `capture_ratio_per_region.csv` |

Nothing else is computed. Every table here is quoted in the response.

### Where this sits in the run

`A3_preflight.ipynb` and the HGCC detection array both run **before** this notebook -- see the
runbook in `README.md`. Everything here reads what they wrote, so it runs **once, top to bottom,
with nothing to adjust**.

**Run this notebook from `R2_revision/ambient_controls/`, on the `mcDETECT-env` kernel** -- §2
imports `mcDETECT` to reproduce Set 2 from Set 1.


## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial import cKDTree

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# THE DEFAULTS BELOW PRODUCE THE FINAL TABLES. Run top to bottom, once, and change nothing.
DRY_RUN = False          # True -> subsample for a cheap smoke pass over every cell. The tables a
                         #   dry run writes are NOT final; set it back to False and rerun.
MAX_SPHERES = 200_000 if DRY_RUN else None

C.ensure_dirs()
OUT = C.A3A_DIR

print("data      :", C.DATA_ROOT)
print("detections:", C.DETECT_DIR)
print("writing to:", OUT)
print("sets      :", C.SETS)
print("thresholds: size_thr <", C.SIZE_THR, "| in_soma <", C.IN_SOMA_THR, "| nc <", C.NC_THR)

data      : /Users/chenyang/Desktop/mcDETECT/data
detections: /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/detect
writing to: /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/a3a
sets      : ['set0', 'set1', 'set2', 'set3']
thresholds: size_thr < 4.0 | in_soma < 0.1 | nc < 0.1


/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

## 1. Inventory and funnels -- how many aggregates does each set produce?

Every set at the same three stages, side by side. The comparison that carries the claim is
not "Set 3 is small" but **Set 3 and Set 0 against the markers, per million transcripts of
the seeding gene, at each stage** -- the sets differ ~15x in abundance and DBSCAN yield is
superlinear in count, so raw counts are not comparable.

`funnel_by_gene.csv` is written per set by `run_detection_sets.py`; this cell pools them.
Set 2 has no funnel: it is the published detection, reused, not re-run.

In [2]:
funnels, inventory = [], []

for set_name in C.SETS:
    for sample in C.SAMPLES:
        p = C.spheres_path(set_name, sample)
        if not p.exists():
            print(f"[skip] {set_name} {sample}: {p} missing")
            continue
        sph = pd.read_parquet(p)
        inventory.append(dict(set=set_name, set_label=C.SET_LABEL[set_name],
                              sample=sample, n_spheres=len(sph),
                              median_sphere_r=float(sph["sphere_r"].median()),
                              median_size=float(sph["size"].median()),
                              median_comp=float(sph["comp"].median()),
                              mean_in_soma=float(sph["in_soma_ratio"].mean())))
        d = C.detect_dir(set_name, sample)
        f = d / "funnel_by_gene.csv"
        if f.exists():
            fg = pd.read_csv(f)
            # A gene that formed NO sphere is absent from the funnel, because
            # flatten_sphere_dict skips empty per-gene frames. run_detection_sets now passes
            # genes= to funnel_counts so future runs emit the zero row, but the tables already
            # on disk predate that -- reindex them here against the list that was actually
            # seeded, which run_info.csv records (post zero-count drop, so it is exactly what
            # DBSCAN saw). Set 3 seeded 18 and reported 16 (WT) / 17 (AD).
            ri = d / "run_info.csv"
            if ri.exists():
                seeded = str(pd.read_csv(ri)["genes"].iloc[0]).split(";")
                absent = [g for g in seeded if g not in set(fg["seed_gene"])]
                if absent:
                    fg = pd.concat([fg, pd.DataFrame([
                        dict(set=set_name, sample=sample, seed_gene=g,
                             **{stage: 0 for stage in C.FUNNEL_STAGES}) for g in absent])],
                        ignore_index=True)
                    print(f"[{set_name} {sample}] {len(absent)} seeded gene(s) formed no sphere "
                          f"-- added as zeros: {absent}")
                fg = (fg.set_index("seed_gene").reindex(seeded).rename_axis("seed_gene")
                        .reset_index())
            funnels.append(fg)

inv = pd.DataFrame(inventory)
inv.to_csv(OUT / "set_inventory.csv", index=False)
display(inv)

if funnels:
    fun = pd.concat(funnels, ignore_index=True)
    # rate per million transcripts of the seeding gene -- neutralises the 15x abundance gap
    counts = {}
    for sample in C.SAMPLES:
        tx = A3.load_transcripts(sample, columns=["target"], verbose=False)
        counts[sample] = tx["target"].value_counts()
        del tx
    fun["n_tx_gene"] = [counts[r.sample].get(r.seed_gene, 0) for r in fun.itertuples()]
    for stage in C.FUNNEL_STAGES:
        fun[f"rate_{stage}_per_Mtx"] = fun[stage] / (fun["n_tx_gene"] / 1e6)
    fun.to_csv(OUT / "funnel_by_gene.csv", index=False)
    display(fun.groupby(["set", "sample"])[[*C.FUNNEL_STAGES,
                                            *[f"rate_{s}_per_Mtx" for s in C.FUNNEL_STAGES]]]
            .sum(numeric_only=True))

[set3 WT] 2 seeded gene(s) formed no sphere -- added as zeros: ['Cyfip1', 'C4a']
[set3 AD] 1 seeded gene(s) formed no sphere -- added as zeros: ['C4a']


,set,set_label,sample,n_spheres,median_sphere_r,median_size,median_comp,mean_in_soma
0,set0,Set 0 (abundance-matched unannotated genes),WT,75452,0.882349,3.0,1.0,0.000531
1,set0,Set 0 (abundance-matched unannotated genes),AD,65919,0.895044,3.0,1.0,0.000257
2,set1,"Set 1 (granule markers, before NC filtering)",WT,741378,0.942654,4.0,2.0,0.000605
3,set1,"Set 1 (granule markers, before NC filtering)",AD,429391,0.961270,4.0,2.0,0.000489
4,set2,Set 2 (published),WT,681337,0.933392,4.0,2.0,0.000500
5,set2,Set 2 (published),AD,398809,0.952230,4.0,2.0,0.000411
6,set3,Set 3 (nuclear-enriched control genes),WT,4305,0.830343,3.0,1.0,0.000058
7,set3,Set 3 (nuclear-enriched control genes),AD,2324,0.787432,3.0,1.0,0.000000


raw     size  in_soma  rate_raw_per_Mtx  rate_size_per_Mtx  \
set  sample                                                                   
set0 AD       143530   143281    66166     360160.675219      359261.162602   
     WT       229669   228910    75847     466469.611196      464677.303077   
set1 AD       660577   659556   437856     460258.242990      459000.304460   
     WT      1184487  1183289   755795     564045.990839      562449.814848   
set3 AD        22172    22172     2324     167126.384280      167126.384280   
     WT        34724    34722     4306     203369.168195      203343.259781   

             rate_in_soma_per_Mtx  
set  sample                        
set0 AD             122158.497196  
     WT             123029.955887  
set1 AD             230959.899209  
     WT             261463.664693  
set3 AD              12364.289332  
     WT              14716.555505

## 2. Overlap -- both directions, a ladder of criteria, leading with the loosest

If structured ambient RNA drove the calls, the control genes would form aggregates sitting on top
of the real granules. So we score **Set 0 and Set 3 against Set 1 and Set 2**, identically.

The Set 1 -> Set 2 comparison is what isolates the negative-control filter, because the two
populations differ in nothing else. That premise is not assumed: re-applying the published filter
to Set 1 reproduces Set 2, and the agreement is reported in `set2_reproduction.csv`.

mcDETECT's own merge predicate (`model.py:349-353`, with `l=1`, `rho=0.2`) is

```
merge(A,B)  <=>  d <= |r_A - r_B|   (containment)   OR   d < 0.2 * (r_A + r_B)
```

i.e. two equal-radius spheres merge only when their centres are within `0.4*r`. That is very
strict -- real granules routinely overlap each other without merging -- so reporting *only* that
predicate would understate co-location and read as rigged. We lead with `intersect`
(`d < r_A + r_B`), the loosest criterion: a small overlap under it is uncontestable.

**Both directions are reported**, because they answer different questions.

* `frac_overlapping` -- the share of **granules** that meet a control aggregate. This is the one
  that bounds contamination of the published result, and it is reported as observed/expected
  against control spheres randomly re-placed in the tissue mask at matched radius and matched
  `layer_z` (20 draws per control set, seeded per sample and per control).
* `frac_control_overlapping` -- the share of **control aggregates** that meet a granule. This is
  the direct form of "the detector already excludes most of this population", and it needs no
  expectation: it is a plain ceiling. `center_in` is asymmetric by construction, so under that
  rung the reverse direction is the mirrored predicate, not the same number read backwards.

**Transcript level as well as granule level.** `merge_sphere` is many-to-one and order-dependent
(its base is `sphere_dict[0]`), so granule-level cardinality is partly an artefact of gene order.
The fraction of in-sphere marker transcripts that also fall inside a control sphere is
merge-invariant, and it is computed for both controls against both granule sets.


In [3]:
overlap_rows, repro_rows = [], []

for sample in C.SAMPLES:
    tx = A3.load_transcripts(sample, columns=["global_x", "global_y", "global_z", "target"])
    mask, xb, yb = A3.tissue_mask(tx)

    # Load both granule sets once; they are re-used by every control and every criterion.
    bases = {}
    for target in C.OVERLAP_BASES:
        pt = C.spheres_path(target, sample)
        if not pt.exists():
            print(f"[skip] {target} {sample} missing")
            continue
        b = pd.read_parquet(pt)
        if MAX_SPHERES:
            b = b.sample(min(MAX_SPHERES, len(b)), random_state=0).reset_index(drop=True)
        bases[target] = b

    for ctrl in C.OVERLAP_CONTROLS:
        pc = C.spheres_path(ctrl, sample)
        if not pc.exists():
            print(f"[skip] {ctrl} {sample} missing")
            continue
        control = pd.read_parquet(pc)

        # Build the null realisations ONCE per (sample, control), before the criterion loop: every
        # criterion must be scored against the SAME 20 placements, or the ladder's rungs are not
        # comparable. The seed carries the control name so Set 0 and Set 3 get independent draws.
        # crc32, not hash(): Python's string hash is salted per process, so hash() would make
        # expected_frac / obs_over_exp differ between kernel restarts.
        rng = np.random.default_rng(
            zlib.crc32(f"{C.OVERLAP_NULL_SEED}|{sample}|{ctrl}".encode()))
        nulls = []
        for _ in range(C.OVERLAP_N_NULL):
            cn = control.copy()
            ok = np.zeros(len(cn), dtype=bool)
            nx, ny = np.zeros(len(cn)), np.zeros(len(cn))
            for _try in range(10):
                todo = ~ok
                if not todo.any():
                    break
                cx = rng.uniform(xb[0], xb[-1], todo.sum())
                cy = rng.uniform(yb[0], yb[-1], todo.sum())
                good = A3.in_tissue(cx, cy, mask, xb, yb)
                idx = np.flatnonzero(todo)[good]
                nx[idx], ny[idx] = cx[good], cy[good]
                ok[idx] = True
            cn["sphere_x"], cn["sphere_y"] = nx, ny
            nulls.append(cn[ok].reset_index(drop=True))
        n_placed = int(np.mean([len(x) for x in nulls])) if nulls else 0
        print(f"[{sample}/{ctrl}] {C.OVERLAP_N_NULL} nulls, mean placed "
              f"{n_placed:,}/{len(control):,}")

        for target, base in bases.items():
            # One pass per (base, control) for the whole ladder -- the candidate query does not
            # depend on the criterion, so scoring the three rungs together costs one pass, not
            # three. Same for the 20 null draws, which is where the time actually goes.
            fwd = A3.overlap_pairs(base, control, criterion=C.OVERLAP_CRITERIA)
            rev = A3.overlap_pairs(control, base, criterion=C.OVERLAP_CRITERIA)
            # BOTH directions get a null. Set 1 holds 741K spheres over 18.8M um^2, so a control
            # sphere dropped anywhere in the tissue has a substantial chance of intersecting one
            # by geometry alone; the control-side fraction is uninterpretable -- and reads far
            # worse than it is -- without the matched expectation beside it.
            null_fwd = [A3.overlap_pairs(base, cn, criterion=C.OVERLAP_CRITERIA) for cn in nulls]
            null_rev = [A3.overlap_pairs(cn, base, criterion=C.OVERLAP_CRITERIA) for cn in nulls]

            for crit in C.OVERLAP_CRITERIA:
                hit = fwd[crit][0]
                back = rev[crit][0]
                null_fracs = [float(nf[crit][0].mean()) for nf in null_fwd]
                null_back = [float(nb_[crit][0].mean()) for nb_ in null_rev]
                obs = float(hit.mean())
                obs_b = float(back.mean())
                exp = float(np.mean(null_fracs)) if null_fracs else np.nan
                exp_b = float(np.mean(null_back)) if null_back else np.nan
                overlap_rows.append(dict(
                    sample=sample, control=ctrl, base=target, criterion=crit,
                    n_base=len(base), n_control=len(control), n_control_placed=n_placed,
                    n_overlapping=int(hit.sum()), frac_overlapping=obs,
                    expected_frac=exp, obs_over_exp=obs / exp if exp else np.nan,
                    null_sd=float(np.std(null_fracs)) if null_fracs else np.nan,
                    n_control_overlapping=int(back.sum()), frac_control_overlapping=obs_b,
                    expected_control_frac=exp_b,
                    obs_over_exp_control=obs_b / exp_b if exp_b else np.nan,
                    null_sd_control=float(np.std(null_back)) if null_back else np.nan,
                    is_primary=(crit == C.OVERLAP_PRIMARY)))

    # The premise of the Set1 -> Set2 comparison, measured rather than asserted: re-applying the
    # published NC filter to Set 1 must give back the published table. Compared on COUNT, not on
    # values -- miniball is randomised, so even two runs of mcDETECT's own fine pass differ by
    # ~1e-12 in the radii, and Set 1 took the rough-then-filter route. What this would catch --
    # a Set 1 not built the way Set 2 was -- shows up as tens of percent.
    p1 = C.spheres_path("set1", sample)
    if p1.exists():
        from mcDETECT.model import mcDETECT
        set1 = pd.read_parquet(p1)
        mc = mcDETECT(transcripts=tx, gnl_genes=C.SYN_GENES,
                      nc_genes=A3.load_nc_genes(sample),      # 19, as published
                      **{k: v for k, v in C.DETECT_KWARGS_FINE.items() if k != "type"},
                      type=C.DETECT_KWARGS_FINE["type"])
        repro = mc.nc_filter(set1)
        pub = pd.read_parquet(C.mcdetect_granules_path(sample))
        repro_rows.append(dict(sample=sample, n_set1=len(set1), n_reproduced=len(repro),
                               n_published=len(pub),
                               rel_diff=(len(repro) - len(pub)) / len(pub)))
        print(f"[{sample}] Set 2 reproduction: {len(repro):,} vs published {len(pub):,} "
              f"({repro_rows[-1]['rel_diff']:+.3%})")
    del tx

ov = pd.DataFrame(overlap_rows)
ov.to_csv(OUT / "overlap_ladder.csv", index=False)
pd.DataFrame(repro_rows).to_csv(OUT / "set2_reproduction.csv", index=False)
display(ov[ov["is_primary"]])

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet


[WT] 103,398,068 transcripts


[WT/set0] 20 nulls, mean placed 75,088/75,452


[WT/set3] 20 nulls, mean placed 4,282/4,305


[WT] Set 2 reproduction: 681,346 vs published 681,337 (+0.001%)
[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet


[AD] 68,876,647 transcripts


[AD/set0] 20 nulls, mean placed 65,828/65,919


[AD/set3] 20 nulls, mean placed 2,320/2,324


[AD] Set 2 reproduction: 398,809 vs published 398,809 (+0.000%)


,sample,control,base,criterion,n_base,n_control,n_control_placed,n_overlapping,frac_overlapping,expected_frac,obs_over_exp,null_sd,n_control_overlapping,frac_control_overlapping,expected_control_frac,obs_over_exp_control,null_sd_control,is_primary
0,WT,set0,set1,intersect,741378,75452,75088,23223,0.031324,0.013867,2.258816,0.000146,16046,0.212665,0.104557,2.033972,0.000965,True
3,WT,set0,set2,intersect,681337,75452,75088,18298,0.026856,0.013651,1.967379,0.000121,13776,0.182580,0.097369,1.875131,0.000840,True
6,WT,set3,set1,intersect,741378,4305,4282,3512,0.004737,0.000683,6.932491,0.000046,1847,0.429036,0.094142,4.557334,0.005097,True
9,WT,set3,set2,intersect,681337,4305,4282,1369,0.002009,0.000669,3.001535,0.000045,936,0.217422,0.086868,2.502895,0.004785,True
12,AD,set0,set1,intersect,429391,65919,65828,11781,0.027437,0.021799,1.258620,0.000226,9013,0.136728,0.106997,1.277870,0.000896,True
15,AD,set0,set2,intersect,398809,65919,65828,8963,0.022474,0.021603,1.040334,0.000211,7359,0.111637,0.099855,1.117991,0.000758,True
18,AD,set3,set1,intersect,429391,2324,2320,851,0.001982,0.000622,3.184284,0.000041,546,0.234940,0.088714,2.648271,0.005149,True
21,AD,set3,set2,intersect,398809,2324,2320,297,0.000745,0.000617,1.207317,0.000042,237,0.101979,0.082618,1.234351,0.005162,True


In [4]:
# Transcript-level overlap -- merge-invariant, so it does not inherit merge_sphere's gene-order
# dependence. Both controls against both granule sets, so the Set1 -> Set2 fall has the same
# backing at transcript level that it has at granule level.
tx_rows = []
for sample in C.SAMPLES:
    tx = A3.load_transcripts(sample)
    marker_tx = tx[tx["target"].isin(C.SYN_GENES)]
    pts = marker_tx[["global_x", "global_y", "global_z"]].to_numpy(float)
    tree = cKDTree(pts)

    def _inside(spheres, z_col="layer_z", chunk=20_000):
        """Batched: one query per chunk of spheres, not one per sphere."""
        flag = np.zeros(len(pts), dtype=bool)
        cen = spheres[["sphere_x", "sphere_y", z_col]].to_numpy(float)
        rad = spheres["sphere_r"].to_numpy(float)
        for lo in range(0, len(cen), chunk):
            hi = min(lo + chunk, len(cen))
            idx = tree.query_ball_point(cen[lo:hi], rad[lo:hi], workers=-1)
            flat = [j for c in idx for j in c]
            if flat:
                flag[np.fromiter(flat, dtype=np.int64, count=len(flat))] = True
        return flag

    inside = {}
    for name in C.OVERLAP_BASES + C.OVERLAP_CONTROLS:
        p = C.spheres_path(name, sample)
        if p.exists():
            inside[name] = _inside(pd.read_parquet(p))

    for target in C.OVERLAP_BASES:
        for ctrl in C.OVERLAP_CONTROLS:
            if target not in inside or ctrl not in inside:
                continue
            ib, ic = inside[target], inside[ctrl]
            tx_rows.append(dict(sample=sample, base=target, control=ctrl,
                                n_marker_tx=len(pts),
                                n_in_base=int(ib.sum()), n_in_control=int(ic.sum()),
                                n_in_both=int((ib & ic).sum()),
                                frac_of_base_also_control=float((ib & ic).sum()
                                                                / max(ib.sum(), 1))))
    del tx, marker_tx, inside

pd.DataFrame(tx_rows).to_csv(OUT / "overlap_transcript_level.csv", index=False)
display(pd.DataFrame(tx_rows))

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet


[WT] 103,398,068 transcripts


[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet


[AD] 68,876,647 transcripts


,sample,base,control,n_marker_tx,n_in_base,n_in_control,n_in_both,frac_of_base_also_control
0,WT,set1,set0,33303096,3412278,146219,43301,0.012690
1,WT,set1,set3,33303096,3412278,13670,5143,0.001507
2,WT,set2,set0,33303096,3051793,146219,34733,0.011381
3,WT,set2,set3,33303096,3051793,13670,1183,0.000388
4,AD,set1,set0,20297113,2035602,74739,17523,0.008608
5,AD,set1,set3,20297113,2035602,4023,980,0.000481
6,AD,set2,set0,20297113,1859967,74739,13025,0.007003
7,AD,set2,set3,20297113,1859967,4023,176,0.000095


## 3. Per-region density, WT vs AD

The reviewer's hypothesis predicts that ambient structure tracks pathology, so it should show the
same regional WT/AD pattern the granules do. Densities are therefore computed per region for all
four sets on each sample's own spot grid.

The statistic that carries the biology is the **per-region AD/WT density ratio**, and Set 1 is a
built-in positive control for it: Set 1 is Set 2 without one filter, so if the ratio profile is
recoverable at all, Set 1 must recover it.

Two things to keep honest. `CAPTURE_EFFICIENCY_COEF` is a **global** scalar and should not be
assumed spatially uniform under structured ambient, so the per-region WT/AD total-transcript ratio
and its spread are reported alongside. And there is one WT and one AD section, so every per-spot
p-value is pseudo-replication -- these numbers are descriptive.


In [5]:
sf = A3._import_sphere_features()
assert sf.C.CAPTURE_EFFICIENCY_COEF == 1.0, (
    "sphere_features' capture coefficient is no longer 1.0 -- A3 applies its own explicitly and "
    "would now double-correct")
density_frames = []

for set_name in ["set0", "set1", "set2", "set3"]:
    for sample in C.SAMPLES:
        p = C.spheres_path(set_name, sample)
        if not p.exists():
            continue
        sph = pd.read_parquet(p)
        spots = sc.read_h5ad(C.spots_path(sample))
        # Two things this call gets wrong if taken at face value:
        #  (1) it returns a TUPLE (density_df, per_spot_df), not a frame;
        #  (2) apply_capture_coef divides by sphere_features' OWN
        #      postproc_config.CAPTURE_EFFICIENCY_COEF, which A1 deliberately set to 1.0 -- so it
        #      is a NO-OP here, not the 0.818691 this analysis means. Apply ours explicitly.
        dens_df, per_spot = sf.subtype_density_per_region(
            sph["sphere_x"].to_numpy(), sph["sphere_y"].to_numpy(),
            np.full(len(sph), "all"), spots, sample,
            apply_capture_coef=False, grid_len=C.SPOT_GRID, n_boot=C.N_BOOTSTRAP)
        if sample == "AD":
            for col in ("density", "sd", "ci_low", "ci_high"):
                if col in dens_df:
                    dens_df[col] = dens_df[col] / C.CAPTURE_EFFICIENCY_COEF
        # subtype_density_per_region emits BOTH an "all" row and an identical "overall" row;
        # keep one or every downstream bar is drawn at twice its true height.
        dens_df = dens_df[dens_df["subtype"] == "overall"].reset_index(drop=True)
        dens_df["set"] = set_name
        density_frames.append(dens_df)

if density_frames:
    dens = pd.concat(density_frames, ignore_index=True)
    # Both samples' spots carry a brain_area == "Unknown" level that AREA_LIST (the published
    # region ordering) does not list. Making the Categorical FIRST would silently turn it into
    # NaN and keep the row with a blank area name -- an unlabelled bar in the R panel. Drop it
    # explicitly and say how many rows went.
    keep = dens["brain_area"].isin(C.AREA_LIST)
    if not keep.all():
        dropped = dens.loc[~keep, "brain_area"].value_counts().to_dict()
        print(f"dropped {int((~keep).sum())} rows outside AREA_LIST: {dropped}")
        dens = dens[keep]
    dens["brain_area"] = pd.Categorical(dens["brain_area"], categories=C.AREA_LIST, ordered=True)
    dens = dens.sort_values(["set", "sample", "brain_area"]).reset_index(drop=True)
    dens.to_csv(OUT / "set_density_per_region.csv", index=False)
    display(dens.head(30))
else:
    print("[skip] no detections on disk yet")

# CAPTURE_EFFICIENCY_COEF is a GLOBAL scalar and must not be assumed spatially uniform under
# structured ambient. Report the per-region WT/AD total-transcript ratio and its spread beside
# any table that uses it, so the reader can see how far the single constant is being stretched.
ratio_rows = []
for sample in C.SAMPLES:
    tx = A3.load_transcripts(sample, columns=["global_x", "global_y"], verbose=False)
    spots = sc.read_h5ad(C.spots_path(sample))
    n = sf.grid_counts(tx["global_x"].to_numpy(), tx["global_y"].to_numpy(), spots,
                       grid_len=C.SPOT_GRID)
    ratio_rows.append(pd.DataFrame({"sample": sample,
                                    "brain_area": spots.obs["brain_area"].to_numpy(),
                                    "n_tx": n}))
    del tx
if ratio_rows:
    rr = pd.concat(ratio_rows, ignore_index=True)
    per_region = rr.groupby(["sample", "brain_area"], observed=True)["n_tx"].sum().unstack(0)
    per_region["AD_over_WT"] = per_region.get("AD") / per_region.get("WT")
    per_region.to_csv(OUT / "capture_ratio_per_region.csv")
    print("global capture coef in use:", C.CAPTURE_EFFICIENCY_COEF)
    print("per-region AD/WT transcript ratio: "
          f"median {per_region['AD_over_WT'].median():.3f}, "
          f"range {per_region['AD_over_WT'].min():.3f}-{per_region['AD_over_WT'].max():.3f}")
    display(per_region)

dropped 8 rows outside AREA_LIST: {'Unknown': 8}


,sample,brain_area,subtype,density,n_spots,density_sd,density_sem,density_ci_low,density_ci_high,set
0,AD,Isocortex,overall,3.431647,2178,3.664497,0.078521,2.663889,2.953237,set0
1,AD,OLF,overall,4.502644,255,3.340813,0.209210,3.301765,4.128333,set0
2,AD,HPF-CA,overall,6.836051,1004,5.096049,0.160830,5.282371,5.890637,set0
3,AD,HPF-DG,overall,3.004712,574,3.369242,0.140629,2.182753,2.736934,set0
4,AD,HPF-SR,overall,1.352074,505,1.657251,0.073747,0.972277,1.267327,set0
5,AD,CTXsp,overall,2.113323,63,2.585358,0.325724,1.102778,2.365079,set0
6,AD,TH,overall,2.707183,208,3.084373,0.213863,1.814784,2.644952,set0
7,AD,MB,overall,17.942645,1987,11.180957,0.250830,14.262506,15.167136,set0
8,AD,FT,overall,15.129233,1735,18.064304,0.433682,11.601614,13.245274,set0
9,WT,Isocortex,overall,6.900601,2495,5.356922,0.107246,6.693838,7.115341,set0


global capture coef in use: 0.818691
per-region AD/WT transcript ratio: median 0.658, range 0.140-4.197


sample,AD,WT,AD_over_WT
brain_area,,,
CTXsp,394656,2817190,0.140089
FT,8101669,11482069,0.705593
HPF-CA,10587990,11178851,0.947145
HPF-DG,6060262,2852528,2.124523
HPF-SR,2605175,8639214,0.301552
Isocortex,21847702,35796052,0.610338
MB,16136937,17548421,0.919566
OLF,1696364,8130673,0.208638
TH,1310331,4919787,0.266339


## Outputs

All under `output/a3a/`. The `preflight/` tables are written by `A3_preflight.ipynb`.

| file | contents | § |
|---|---|---|
| `set_inventory.csv` | spheres per set and sample, with median radius, size and in-soma ratio | 1 |
| `funnel_by_gene.csv` | raw -> size -> in-soma per seed gene, and the per-million-transcript rates | 1 |
| `overlap_ladder.csv` | both controls x both granule sets x three criteria, both directions, with the re-placement null | 2 |
| `overlap_transcript_level.csv` | the same overlap at transcript level, merge-invariant | 2 |
| `set2_reproduction.csv` | re-applying the published NC filter to Set 1 recovers Set 2 | 2 |
| `set_density_per_region.csv` | objects per spot by region and set, WT and AD | 3 |
| `capture_ratio_per_region.csv` | per-region AD/WT total-transcript ratio -- how far the single global capture scalar is stretched | 3 |

Every one of these is quoted in the response. Nothing is written "for the record".
